In [1]:
import os
import torch

num_cores = os.cpu_count()
num_GPUs = torch.cuda.device_count()

print('# Cores:' + str(num_cores))
print('# GPUs: ' + str(num_GPUs))

# Get the available GPUs directly as a list
print(f"Available GPUs: {list(range(torch.cuda.device_count()))}")

print('Visible GPUs Indices: ' + str(os.environ.get('CUDA_VISIBLE_DEVICES', 'All GPUs are visible')))

# Cores:8
# GPUs: 0
Available GPUs: []
Visible GPUs Indices: All GPUs are visible


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

In [3]:
%load_ext autoreload
%autoreload 2

import random
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

from utils.load_networkx import networkx_feat
from utils.macro_dataset import MacroDataset
from utils.macro_supervised_hyperparameter import MacroSupervised
from utils.macro_attribution import Attribution
from utils import plot

In [4]:
# Same
DESCRIPTORS = 'unique_descriptors.json'

SEED = 112
TASK = 'classification'
MODEL = 'AttentiveFP'
LABELNAME = 'immunogenic'

NUM_EPOCHS = 10
NUM_WORKERS = num_cores

# MODEL_PATH = 'past_trials/' + MODEL + '/hyperparameter_optimization' 
MODEL_PATH = 'past_trials/' + MODEL + '/' + str(NUM_EPOCHS) + '_epochs'


SAVE_MODEL = True
SAVE_OPT = True
SAVE_CONFIG = True

CUSTOM_PARAMS = {}

In [5]:
# Train, Val, Test
MON_SMILES = 'tables/SMILES_peptides_monomer.txt'
BOND_SMILES = 'tables/SMILES_peptides_bond.txt'
TXT_DATA_PATH = 'dataset/classification/'
DF_PATH = 'tables/immuno_peptides.txt'

SPLIT = '0.7,0.3'
SCALER_TYPE = 'MinMax'

NX_GRAPHS, SCALERS = networkx_feat(
    TXT_DATA_PATH = TXT_DATA_PATH, 
    MON_SMILES = MON_SMILES, 
    BOND_SMILES = BOND_SMILES, 
    DESCRIPTORS = DESCRIPTORS,
    SPLIT = SPLIT,
    SEED = SEED,
    SCALER_TYPE = SCALER_TYPE
)

dgl_dict = MacroDataset(
    DF_PATH = DF_PATH,
    TASK = TASK, 
    LABELNAME = LABELNAME, 
    MODEL = MODEL,
    NX_GRAPHS = NX_GRAPHS)

In [7]:
import pickle
with open('scalers.pkl', 'rb') as file:
    data = pickle.load(file)

UnpicklingError: invalid load key, '\x01'.

In [24]:
# Inference
MON_SMILES = 'tables_poly/SMILES_polymers_monomer.txt'
BOND_SMILES = 'tables_poly/SMILES_polymers_bond.txt'
TXT_DATA_PATH = 'shoshana_polymers/dataset_uniform/classification'
DF_PATH = 'tables_poly/immuno_polymers.txt'

NX_GRAPHS_INFER = networkx_feat(
    TXT_DATA_PATH = TXT_DATA_PATH, 
    MON_SMILES = MON_SMILES, 
    BOND_SMILES = BOND_SMILES, 
    DESCRIPTORS = DESCRIPTORS,
    SCALER = SCALERS
)

dgl_dict_infer = MacroDataset(
    DF_PATH = DF_PATH,
    TASK = TASK, 
    LABELNAME = LABELNAME, 
    MODEL = MODEL,
    NX_GRAPHS = NX_GRAPHS_INFER)

In [7]:
# plot.graph(NX_GRAPHS_INFER['inference']['SID2'])
# NX_GRAPHS_INFER

In [8]:
macro_supervised = MacroSupervised(MacroDataset = dgl_dict, MODEL = MODEL, DESCRIPTORS = DESCRIPTORS, NUM_EPOCHS = NUM_EPOCHS,
                                   NUM_WORKERS = NUM_WORKERS, CUSTOM_PARAMS = {}, INFERENCE = dgl_dict_infer,
                                   MODEL_PATH = MODEL_PATH, SAVE_MODEL = SAVE_MODEL, SAVE_OPT = SAVE_OPT,
                                   SAVE_CONFIG = SAVE_CONFIG)

Directory past_trials/AttentiveFP/1_epochs already exists.


In [9]:
macro_supervised.main()

1
tensor([[0.1747],
        [0.2660],
        [0.3667],
        [0.2833],
        [0.2078],
        [0.4173],
        [0.4092],
        [0.1719],
        [0.2704],
        [0.3929],
        [0.2530],
        [0.3575],
        [0.3223],
        [0.4178],
        [0.2065],
        [0.3102],
        [0.4408],
        [0.2675],
        [0.2868],
        [0.3124],
        [0.1906],
        [0.1977],
        [0.2345],
        [0.4869],
        [0.3524],
        [0.1735],
        [0.3907],
        [0.2926],
        [0.3268],
        [0.3335],
        [0.3382],
        [0.4321],
        [0.4053],
        [0.3538],
        [0.2694],
        [0.3493],
        [0.2807],
        [0.3836],
        [0.2890],
        [0.3944],
        [0.4325],
        [0.1622],
        [0.4552],
        [0.5031],
        [0.3125],
        [0.6177],
        [0.3356],
        [0.4102],
        [0.3929],
        [0.4032],
        [0.2897],
        [0.4101],
        [0.2595],
        [0.1841],
        [0.2432],
        

-0.6549707602339182

# Bayesian Hyperparameter Optimization

## Optuna

In [10]:
import optuna
import optunahub
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution

In [11]:
# Load the HEBO module dynamically
module = optunahub.load_module("samplers/hebo")
HEBOSampler = module.HEBOSampler

# Define the search space using Optuna distributions
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution

search_space = {
    'lr': FloatDistribution(1e-5, 1e-2),
    'weight_decay': FloatDistribution(0.0, 1e-3),
    'num_layers': IntDistribution(1, 5),
    'num_timesteps': IntDistribution(1, 8),
    'graph_feat_size': CategoricalDistribution([16, 32, 64, 128, 256]),
    'dropout': FloatDistribution(0.0, 0.5),
}

# Initialize the HEBOSampler with the defined search space
sampler = HEBOSampler(search_space)

# Define the path for your database
db_directory = 'past_trials/' + str(MODEL)
db_path = os.path.join(db_directory, str(MODEL) + '_hpo.db')
db_url = f'sqlite:///{db_path}'

# Create the directory if it doesn't exist
os.makedirs(db_directory, exist_ok=True)

# Create a study object with the SQLite storage
study = optuna.create_study(
    study_name="hebo",
    sampler=sampler,
    direction="minimize",
    storage=db_url,  # Use SQLite as the storage backend
    load_if_exists=True  # Load the study if it already exists
)

[I 2024-09-03 10:20:34,406] A new study created in RDB with name: hebo


In [12]:
def objective(trial):
    # The parameters are automatically suggested based on the search space
    lr = trial.suggest_float('lr', 1e-5, 1e-2)
    weight_decay = trial.suggest_float('weight_decay', 0.0, 1e-3)
    num_layers = trial.suggest_int('num_layers', 1, 5)
    num_timesteps = trial.suggest_int('num_timesteps', 1, 8)
    graph_feat_size = trial.suggest_categorical('graph_feat_size', [16, 32, 64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.0, 0.5)

    params = {
        'lr': lr,
        'weight_decay': weight_decay,
        'num_layers': num_layers,
        'num_timesteps': num_timesteps,
        'graph_feat_size': graph_feat_size,
        'dropout': dropout,
    }

    print(params)

    new_path = MODEL_PATH + '/model_' + str(trial.number)
    macro_supervised = MacroSupervised(MacroDataset=dgl_dict,
                                       MODEL=MODEL, 
                                       NUM_EPOCHS=NUM_EPOCHS, 
                                       NUM_WORKERS=NUM_WORKERS,
                                       DESCRIPTORS = DESCRIPTORS,
                                       CUSTOM_PARAMS=params,
                                       INFERENCE=dgl_dict_infer,
                                       MODEL_PATH=new_path, 
                                       SAVE_MODEL=SAVE_MODEL, 
                                       SAVE_OPT=SAVE_OPT, 
                                       SAVE_CONFIG=SAVE_CONFIG)

    return macro_supervised.main()


In [13]:
# Optimize the study
study.optimize(objective, n_trials=3)

{'lr': 0.006650581955909729, 'weight_decay': 0.0007833067793399096, 'num_layers': 3, 'num_timesteps': 2, 'graph_feat_size': 64, 'dropout': 0.30716150999069214}
Created directory past_trials/AttentiveFP/hyperparameter_optimization/model_0
1
2
3


[I 2024-09-03 10:22:33,008] Trial 0 finished with value: -0.7567567567567568 and parameters: {'lr': 0.006650581955909729, 'weight_decay': 0.0007833067793399096, 'num_layers': 3, 'num_timesteps': 2, 'graph_feat_size': 64, 'dropout': 0.30716150999069214}. Best is trial 0 with value: -0.7567567567567568.


{'lr': 0.004903451539576054, 'weight_decay': 0.0004421150078997016, 'num_layers': 4, 'num_timesteps': 6, 'graph_feat_size': 128, 'dropout': 0.15918761491775513}
Created directory past_trials/AttentiveFP/hyperparameter_optimization/model_1
1
2
3


[I 2024-09-03 10:24:24,748] Trial 1 finished with value: -0.7567567567567568 and parameters: {'lr': 0.004903451539576054, 'weight_decay': 0.0004421150078997016, 'num_layers': 4, 'num_timesteps': 6, 'graph_feat_size': 128, 'dropout': 0.15918761491775513}. Best is trial 0 with value: -0.7567567567567568.


{'lr': 0.002013971796259284, 'weight_decay': 0.0006130062392912805, 'num_layers': 1, 'num_timesteps': 3, 'graph_feat_size': 64, 'dropout': 0.033072128891944885}
Created directory past_trials/AttentiveFP/hyperparameter_optimization/model_2
1
2
3


[I 2024-09-03 10:26:13,644] Trial 2 finished with value: 0.0 and parameters: {'lr': 0.002013971796259284, 'weight_decay': 0.0006130062392912805, 'num_layers': 1, 'num_timesteps': 3, 'graph_feat_size': 64, 'dropout': 0.033072128891944885}. Best is trial 0 with value: -0.7567567567567568.


In [14]:
# Get the best parameters
best_params = study.best_params
print(f"Best parameters: {best_params}")

Best parameters: {'lr': 0.006650581955909729, 'weight_decay': 0.0007833067793399096, 'num_layers': 3, 'num_timesteps': 2, 'graph_feat_size': 64, 'dropout': 0.30716150999069214}


In [ ]:
# IN: gv5
# optuna-dashboard sqlite:///optuna_study.db
# USE: ` find . -name "*.db" ` TO FIND ALL .db files

## Skopt

In [ ]:
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args

In [ ]:
# ATTENTIVEFP
# {
#   "lr": 5e-3,
#   "weight_decay": 1.87858e-4,
#   "patience": 30,
#   "batch_size": 256,
#   "num_layers": 3,
#   "num_timesteps": 1,
#   "graph_feat_size": 16,
#   "dropout": 0.18742
# }

# Categorical([2, 3], name='hidden_layers')

space = [
    Real(1e-5, 1e-2, name='lr'),               # Learning Rate
    Real(0, 1e-3, name='weight_decay'),        # Weight Decay
    Integer(1, 5, name='num_layers'),          # Number of GNN layers
    Integer(1, 8, name='num_timesteps'),       # Times of updating the graph representations with GRU.
    Categorical([16, 32, 64, 128, 256], name='graph_feat_size'),   # Size for the learned graph representations.
    Real(0.1, 0.5, name='dropout'),            # Dropout Rate
]

In [ ]:
iteration = 1

@use_named_args(space)
def objective(**params):
    global iteration
    print(params)

    new_path = MODEL_PATH + '/model_' + str(iteration)
    macro_supervised = MacroSupervised(MacroDataset = dgl_dict, MODEL = MODEL, NUM_EPOCHS = NUM_EPOCHS,
                                       NUM_WORKERS = NUM_WORKERS, DESCRIPTORS = DESCRIPTORS, CUSTOM_PARAMS = params,
                                       INFERENCE = dgl_dict_infer, MODEL_PATH = new_path, SAVE_MODEL = SAVE_MODEL, SAVE_OPT = SAVE_OPT,
                                       SAVE_CONFIG = SAVE_CONFIG)
    iteration += 1
    return macro_supervised.main()

In [ ]:
result = gp_minimize(
    func=objective,
    dimensions=space,
    n_calls=10,
    random_state=42
)

In [ ]:
result.x

In [ ]:
# MPNN 
{
  "lr": 4.79060e-3,
  "weight_decay": 0,
  "patience": 30,
  "batch_size": 256,
  "node_out_feats": 64,           # Size for the output node representations. Default to 64.
  "edge_hidden_feats": 128,       # Size for the hidden edge representations. Default to 128.
  "num_step_message_passing": 6,  # Number of message passing steps. Default to 6.
  "num_step_set2set": 3,          # Number of set2set steps.
  "num_layer_set2set": 3          # Number of set2set layers.
}


{
    'node_out_feats': CategoricalDistribution([16, 32, 64, 128, 256]),
    'edge_hidden_feats': CategoricalDistribution([16, 32, 64, 128, 256]),
    'num_step_message_passing': IntDistribution(3, 9)
    'num_step_set2set': IntDistribution(1, 6)
    'num_layer_set2set': IntDistribution(1, 6)
}

In [ ]:
# Weave
{
  "lr": 5.09056e-4,
  "weight_decay": 7.04655e-5,
  "patience": 30,
  "batch_size": 256,
  "num_gnn_layers": 4,        # Number of GNN (Weave) layers to use. Default to 2.
  "gnn_hidden_feats": 128,    # Size for the hidden node and edge representations. Default to 50.
  "graph_feats": 16,          # Size for the hidden graph representations. Default to 50.
  "gaussian_expand": false    # Whether to expand each dimension of node features by gaussian histogram in computing graph representations. Default to True.
}


{
    "num_gnn_layers": IntDistribution(2, 6),
    "gnn_hidden_feats": CategoricalDistribution([16, 32, 64, 128, 256]),
    "graph_feats": CategoricalDistribution([8, 16, 32, 64, 128]),
    
}

In [ ]:
# GAT
{
  "lr": 4.78612e-3,
  "weight_decay": 0,
  "patience": 30,
  "batch_size": 256,
  "dropout": 5.627886e-2,
  "gnn_hidden_feats": 32,         # LIST: ``hidden_feats[i]`` gives the output size of an attention head in the i-th GAT layer.
  "num_heads": 4,                 # LIST: ``num_heads[i]`` gives the number of attention heads in the i-th GAT layer.
  "alpha": 0.69093,               # LIST: ``alphas[i]`` gives the slope for negative value in the i-th GAT layer.
  "predictor_hidden_feats": 128,  # Size for hidden representations in the output MLP predictor. Default to 128.
  "num_gnn_layers": 4,            # ALL LISTS ARE OF THIS LENGTH
  "residual": true                # LIST: ``residual[i]`` decides if residual connection is to be used for the i-th GAT layer.
}

{
    "gnn_hidden_feats": 32,         # LIST: ``hidden_feats[i]`` gives the output size of an attention head in the i-th GAT layer.
    "num_heads": 4,                 # LIST: ``num_heads[i]`` gives the number of attention heads in the i-th GAT layer.
    "predictor_hidden_feats": 128,  # Size for hidden representations in the output MLP predictor. Default to 128.
    "num_gnn_layers": 4,            # ALL LISTS ARE OF THIS LENGTH
    
}



    'num_layers': IntDistribution(1, 5),
    'num_timesteps': IntDistribution(1, 8),
    'graph_feat_size': CategoricalDistribution([16, 32, 64, 128, 256]),


In [ ]:
# GCN
{
  "lr": 1.68653e-3,
  "weight_decay": 0,
  "patience": 30,
  "batch_size": 256,
  "dropout": 8.64073e-2,
  "gnn_hidden_feats": 256,      # LIST: ``hidden_feats[i]`` gives the size of node representations after the i-th GCN layer.
  "predictor_hidden_feats": 64, # Size for hidden representations in the output MLP predictor. Default to 128.
  "num_gnn_layers": 5,          # ALL LISTS ARE OF THIS LENGTH
  "residual": true,             # LIST: ``residual[i]`` decides if residual connection is to be used for the i-th GCN layer.
  "batchnorm": true             # LIST: ``batchnorm[i]`` decides if batch normalization is to be applied on the output of the i-th GCN layer.
}



    # gnn_norm : list of str
    #     ``gnn_norm[i]`` gives the message passing normalizer for the i-th GCN layer, which
    #     can be `'right'`, `'both'` or `'none'`. The `'right'` normalizer divides the aggregated
    #     messages by each node's in-degree. The `'both'` normalizer corresponds to the symmetric
    #     adjacency normalization in the original GCN paper. The `'none'` normalizer simply sums
    #     the messages. ``len(gnn_norm)`` equals the number of GCN layers. By default, we use
    #     ``['none', 'none']``.